# 01 - NRL Data Exploration

This notebook explores the raw and processed datasets for the NRL Match Winner Prediction project.
We examine matches, lineups, odds, and ladder data to understand data quality, distributions,
and key patterns before feature engineering and modelling.

**Sections:**
1. Load processed data
2. Dataset overview (shape, dtypes, sample rows)
3. Basic statistics (matches per season, teams, score distributions)
4. Home win rate by season
5. Score distribution histograms
6. Average attendance by venue
7. Missing data analysis
8. Team performance summary

In [ ]:
# ============================================================
# Cell 1: Imports and load processed parquet files
# ============================================================
import sys
from pathlib import Path

# Add the project root to sys.path so we can import project modules
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

from config.settings import PROCESSED_DIR, RAW_DIR, START_YEAR, END_YEAR

# Plot styling
sns.set_theme(style="whitegrid", palette="deep", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 120
warnings.filterwarnings("ignore", category=FutureWarning)

# ------------------------------------------------------------------
# Load processed parquet files
# If processed files do not exist yet, fall back to loading from raw
# CSVs and running the cleaning pipeline.
# ------------------------------------------------------------------
def load_parquet_or_warn(name: str) -> pd.DataFrame:
    """Try loading from PROCESSED_DIR; return empty DataFrame on failure."""
    path = PROCESSED_DIR / f"{name}.parquet"
    if path.exists():
        df = pd.read_parquet(path)
        print(f"Loaded {name}: {df.shape[0]:,} rows x {df.shape[1]} cols from {path.name}")
        return df
    # Try CSV fallback
    csv_path = PROCESSED_DIR / f"{name}.csv"
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        print(f"Loaded {name}: {df.shape[0]:,} rows x {df.shape[1]} cols from {csv_path.name}")
        return df
    print(f"WARNING: {name} not found in {PROCESSED_DIR}")
    return pd.DataFrame()

matches = load_parquet_or_warn("matches")
lineups = load_parquet_or_warn("lineups")
odds = load_parquet_or_warn("odds")
ladders = load_parquet_or_warn("ladders")

print(f"\nProject root: {PROJECT_ROOT}")
print(f"Data directory: {PROCESSED_DIR}")
print(f"Season range: {START_YEAR} - {END_YEAR}")

In [ ]:
# ============================================================
# Cell 2: Dataset shape, dtypes, and sample rows
# ============================================================

datasets = {"matches": matches, "lineups": lineups, "odds": odds, "ladders": ladders}

for name, df in datasets.items():
    if df.empty:
        print(f"\n{'='*60}\n{name.upper()} -- EMPTY\n{'='*60}")
        continue
    print(f"\n{'='*60}")
    print(f"{name.upper()} -- {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(f"{'='*60}")
    print(f"\nColumn types:")
    print(df.dtypes.to_string())
    print(f"\nFirst 3 rows:")
    display(df.head(3))

In [ ]:
# ============================================================
# Cell 3: Basic stats - matches per season, teams per season,
#          score distributions
# ============================================================

if not matches.empty:
    # Determine season column (might be 'season' or 'year')
    season_col = "season" if "season" in matches.columns else "year"

    print("MATCHES PER SEASON")
    print("-" * 40)
    season_counts = matches[season_col].value_counts().sort_index()
    print(season_counts.to_string())
    print(f"\nTotal matches: {len(matches):,}")
    print(f"Average per season: {season_counts.mean():.1f}")

    # Teams per season
    print(f"\nTEAMS PER SEASON")
    print("-" * 40)
    teams_per_season = (
        matches.groupby(season_col)
        .apply(lambda g: pd.concat([g["home_team"], g["away_team"]]).nunique())
    )
    print(teams_per_season.to_string())

    # Score distributions
    if "home_score" in matches.columns and "away_score" in matches.columns:
        print(f"\nSCORE DISTRIBUTIONS")
        print("-" * 40)
        score_stats = matches[["home_score", "away_score"]].describe()
        print(score_stats.to_string())

        matches["total_score"] = matches["home_score"] + matches["away_score"]
        matches["margin"] = (matches["home_score"] - matches["away_score"]).abs()
        print(f"\nAverage total score: {matches['total_score'].mean():.1f}")
        print(f"Median margin: {matches['margin'].median():.0f}")
        print(f"Matches decided by <= 6 points: {(matches['margin'] <= 6).sum()} "
              f"({(matches['margin'] <= 6).mean() * 100:.1f}%)")
else:
    print("Matches data not available.")

In [ ]:
# ============================================================
# Cell 4: Home win rate by season (bar chart)
# ============================================================

if not matches.empty and "home_score" in matches.columns:
    season_col = "season" if "season" in matches.columns else "year"

    # Home win = home_score > away_score (exclude draws)
    matches["home_win"] = (matches["home_score"] > matches["away_score"]).astype(int)

    home_wr_by_season = (
        matches.groupby(season_col)["home_win"]
        .mean()
        .sort_index()
    )

    fig, ax = plt.subplots(figsize=(14, 6))
    bars = ax.bar(
        home_wr_by_season.index.astype(str),
        home_wr_by_season.values,
        color=sns.color_palette("Blues_d", len(home_wr_by_season)),
        edgecolor="white",
        linewidth=0.8,
    )

    # Reference line at 50%
    ax.axhline(0.5, color="red", linestyle="--", linewidth=1, label="50% baseline")

    # Overall average
    overall_hr = matches["home_win"].mean()
    ax.axhline(overall_hr, color="green", linestyle="-.", linewidth=1,
               label=f"Overall avg: {overall_hr:.1%}")

    # Annotate each bar with the percentage
    for bar, val in zip(bars, home_wr_by_season.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{val:.1%}",
            ha="center", va="bottom", fontsize=8, fontweight="bold",
        )

    ax.set_xlabel("Season")
    ax.set_ylabel("Home Win Rate")
    ax.set_title("NRL Home Win Rate by Season", fontsize=14, fontweight="bold")
    ax.set_ylim(0, 0.80)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
    ax.legend(loc="upper right")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    print(f"\nOverall home win rate: {overall_hr:.3f} ({overall_hr:.1%})")
else:
    print("Insufficient data for home win rate analysis.")

In [ ]:
# ============================================================
# Cell 5: Score distribution histograms (home vs away)
# ============================================================

if not matches.empty and "home_score" in matches.columns:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: Home vs Away score distributions (overlaid)
    ax = axes[0]
    bins = np.arange(0, 65, 2)
    ax.hist(matches["home_score"].dropna(), bins=bins, alpha=0.6,
            label="Home", color="#1f77b4", edgecolor="white")
    ax.hist(matches["away_score"].dropna(), bins=bins, alpha=0.6,
            label="Away", color="#ff7f0e", edgecolor="white")
    ax.axvline(matches["home_score"].mean(), color="#1f77b4",
               linestyle="--", linewidth=1.5, label=f"Home mean: {matches['home_score'].mean():.1f}")
    ax.axvline(matches["away_score"].mean(), color="#ff7f0e",
               linestyle="--", linewidth=1.5, label=f"Away mean: {matches['away_score'].mean():.1f}")
    ax.set_xlabel("Score")
    ax.set_ylabel("Frequency")
    ax.set_title("Home vs Away Score Distribution")
    ax.legend(fontsize=8)

    # Panel 2: Total score distribution
    ax = axes[1]
    total_scores = matches["home_score"] + matches["away_score"]
    ax.hist(total_scores.dropna(), bins=30, color="#2ca02c", alpha=0.7, edgecolor="white")
    ax.axvline(total_scores.mean(), color="red", linestyle="--",
               label=f"Mean: {total_scores.mean():.1f}")
    ax.set_xlabel("Total Match Score")
    ax.set_ylabel("Frequency")
    ax.set_title("Total Score Distribution")
    ax.legend()

    # Panel 3: Winning margin distribution
    ax = axes[2]
    margin = (matches["home_score"] - matches["away_score"]).dropna()
    ax.hist(margin, bins=np.arange(-50, 55, 4), color="#9467bd",
            alpha=0.7, edgecolor="white")
    ax.axvline(0, color="red", linestyle="-", linewidth=1.5, label="Draw")
    ax.axvline(margin.mean(), color="green", linestyle="--",
               label=f"Mean: {margin.mean():+.1f} (home advantage)")
    ax.set_xlabel("Margin (Home - Away)")
    ax.set_ylabel("Frequency")
    ax.set_title("Winning Margin Distribution")
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()
else:
    print("Score data not available for histogram analysis.")

In [ ]:
# ============================================================
# Cell 6: Average attendance by venue (top 15)
# ============================================================

if not matches.empty and "attendance" in matches.columns and "venue" in matches.columns:
    # Drop rows where attendance or venue is missing
    att_df = matches.dropna(subset=["attendance", "venue"]).copy()
    att_df["attendance"] = pd.to_numeric(att_df["attendance"], errors="coerce")
    att_df = att_df.dropna(subset=["attendance"])

    venue_stats = (
        att_df.groupby("venue")["attendance"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "avg_attendance", "count": "n_matches"})
        .query("n_matches >= 5")  # Only venues with at least 5 matches
        .sort_values("avg_attendance", ascending=False)
        .head(15)
    )

    fig, ax = plt.subplots(figsize=(12, 7))
    bars = ax.barh(
        venue_stats.index[::-1],
        venue_stats["avg_attendance"][::-1],
        color=sns.color_palette("viridis", 15),
        edgecolor="white",
    )

    # Annotate with match count
    for bar, n_matches in zip(bars, venue_stats["n_matches"][::-1]):
        ax.text(
            bar.get_width() + 200,
            bar.get_y() + bar.get_height() / 2,
            f"  ({n_matches} games)",
            va="center", fontsize=8, color="gray",
        )

    ax.set_xlabel("Average Attendance")
    ax.set_title("Top 15 NRL Venues by Average Attendance", fontsize=14, fontweight="bold")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f"{x:,.0f}"))
    plt.tight_layout()
    plt.show()

    print(f"\nOverall average attendance: {att_df['attendance'].mean():,.0f}")
    print(f"Median attendance: {att_df['attendance'].median():,.0f}")
else:
    print("Attendance or venue data not available.")
    print(f"Available columns: {list(matches.columns) if not matches.empty else 'N/A'}")

In [ ]:
# ============================================================
# Cell 7: Missing data analysis - heatmap of nulls
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle("Missing Data Analysis Across All Datasets", fontsize=16, fontweight="bold", y=1.01)

for ax, (name, df) in zip(axes.ravel(), datasets.items()):
    if df.empty:
        ax.text(0.5, 0.5, f"{name}\n(Empty)", ha="center", va="center",
                fontsize=14, transform=ax.transAxes)
        ax.set_title(f"{name.upper()} - No Data")
        continue

    # Calculate percentage of missing values per column
    missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

    # Only show columns that have some missing data, or top 25 cols
    cols_with_missing = missing_pct[missing_pct > 0]
    if len(cols_with_missing) == 0:
        ax.text(0.5, 0.5, f"{name}\nNo missing values!", ha="center", va="center",
                fontsize=12, transform=ax.transAxes, color="green")
        ax.set_title(f"{name.upper()} - {df.shape[1]} columns, 0% missing")
        continue

    show_cols = cols_with_missing.head(20)

    # Create a heatmap-style visualization
    missing_matrix = df[show_cols.index].isnull().astype(int)

    # Summarize as a bar chart for clarity
    colors = ["#d62728" if v > 50 else "#ff7f0e" if v > 20 else "#2ca02c" for v in show_cols.values]
    ax.barh(range(len(show_cols)), show_cols.values, color=colors, edgecolor="white")
    ax.set_yticks(range(len(show_cols)))
    ax.set_yticklabels(show_cols.index, fontsize=8)
    ax.set_xlabel("Missing %")
    ax.set_title(f"{name.upper()} - Missing Data ({len(cols_with_missing)}/{df.shape[1]} cols)")
    ax.invert_yaxis()

    # Annotate percentages
    for i, v in enumerate(show_cols.values):
        ax.text(v + 0.5, i, f"{v:.1f}%", va="center", fontsize=7)

plt.tight_layout()
plt.show()

# Print summary table
print("\nMISSING DATA SUMMARY")
print("=" * 50)
for name, df in datasets.items():
    if df.empty:
        continue
    total_cells = df.shape[0] * df.shape[1]
    missing_cells = df.isnull().sum().sum()
    pct = missing_cells / total_cells * 100 if total_cells > 0 else 0
    print(f"{name:12s}: {missing_cells:>8,} / {total_cells:>10,} cells missing ({pct:.2f}%)")

In [ ]:
# ============================================================
# Cell 8: Team performance summary across all seasons
# ============================================================

if not matches.empty and "home_score" in matches.columns:
    # Build a record for each team: wins, losses, draws, total matches
    records = []

    for _, row in matches.iterrows():
        hs = row.get("home_score")
        as_ = row.get("away_score")
        ht = row.get("home_team")
        at = row.get("away_team")

        if pd.isna(hs) or pd.isna(as_) or pd.isna(ht) or pd.isna(at):
            continue

        # Home team record
        records.append({
            "team": ht,
            "wins": 1 if hs > as_ else 0,
            "losses": 1 if hs < as_ else 0,
            "draws": 1 if hs == as_ else 0,
            "points_for": hs,
            "points_against": as_,
        })
        # Away team record
        records.append({
            "team": at,
            "wins": 1 if as_ > hs else 0,
            "losses": 1 if as_ < hs else 0,
            "draws": 1 if hs == as_ else 0,
            "points_for": as_,
            "points_against": hs,
        })

    records_df = pd.DataFrame(records)
    team_summary = (
        records_df.groupby("team")
        .agg(
            played=("wins", "count"),
            wins=("wins", "sum"),
            losses=("losses", "sum"),
            draws=("draws", "sum"),
            points_for=("points_for", "sum"),
            points_against=("points_against", "sum"),
        )
        .assign(
            win_pct=lambda d: d["wins"] / d["played"],
            point_diff=lambda d: d["points_for"] - d["points_against"],
            ppg_for=lambda d: d["points_for"] / d["played"],
            ppg_against=lambda d: d["points_against"] / d["played"],
        )
        .sort_values("win_pct", ascending=False)
    )

    print("TEAM PERFORMANCE SUMMARY (All Seasons)")
    print("=" * 80)
    display(
        team_summary[
            ["played", "wins", "losses", "draws", "win_pct", "point_diff", "ppg_for", "ppg_against"]
        ].style.format(
            {
                "win_pct": "{:.3f}",
                "point_diff": "{:+.0f}",
                "ppg_for": "{:.1f}",
                "ppg_against": "{:.1f}",
            }
        ).background_gradient(subset=["win_pct"], cmap="RdYlGn")
    )

    # Visualize as horizontal bar chart
    fig, ax = plt.subplots(figsize=(12, 8))
    team_sorted = team_summary.sort_values("win_pct", ascending=True)
    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(team_sorted)))
    ax.barh(team_sorted.index, team_sorted["win_pct"], color=colors, edgecolor="white")
    ax.axvline(0.5, color="red", linestyle="--", linewidth=1, label="50%")
    ax.set_xlabel("Win Rate")
    ax.set_title("NRL Team Win Rates (All Seasons)", fontsize=14, fontweight="bold")
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
    ax.legend()

    # Annotate
    for i, (team, row) in enumerate(team_sorted.iterrows()):
        ax.text(row["win_pct"] + 0.005, i, f"{row['win_pct']:.1%} ({row['played']})",
                va="center", fontsize=8)

    plt.tight_layout()
    plt.show()
else:
    print("Insufficient data for team performance summary.")